In [ ]:
import cv2
import os
import json
import glob
import numpy as np
import re
from collections import defaultdict
import random
import matplotlib.pyplot as plt

# Configuración de rutas y parámetros
test_images_dir = "../data/test_images/"
png_dir = "../data/bill_processing/processed_images/"
database_dir = "../data/full_bill_database/"
RATIO_THRESH = 0.7
THRESHOLDS_FILE = 'thresholds.json'

def get_category_from_db_filename(filename):
    return filename.replace("clean_", "").replace(".png", "")

def get_denomination_from_category(category):
    match = re.search(r'\d+', category)
    if match:
        denom = match.group()
        if "Polimero" in category:
            return denom + "Polimero"
        return denom
    return category

def load_dataset(test_images_dir):
    dataset = defaultdict(list)
    for root, dirs, files in os.walk(test_images_dir):
        category = os.path.basename(root)
        if not category.isdigit():
            continue
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                filepath = os.path.join(root, file)
                dataset[category].append(filepath)
    return dataset

def split_dataset(dataset, train_ratio=0.75):
    train_set = defaultdict(list)
    test_set = defaultdict(list)
    for category, images in dataset.items():
        sorted_images = sorted(images)
        random.seed(42)
        random.shuffle(sorted_images)
        split_idx = max(1, int(len(sorted_images) * train_ratio))
        if len(sorted_images) == 1:
            train_set[category] = sorted_images
            test_set[category] = sorted_images
        else:
            train_set[category] = sorted_images[:split_idx]
            test_set[category] = sorted_images[split_idx:]
    return train_set, test_set

def extract_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None, None
        
    max_dim = 1024
    h, w = img.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create(nfeatures=5000)
    kp, des = sift.detectAndCompute(gray, None)
    return kp, des

def count_good_matches(des_query, des_db):
    if des_query is None or des_db is None or len(des_query) < 2 or len(des_db) < 2:
        return 0
    matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    matches = matcher.knnMatch(des_query, des_db, k=2)
    return sum(1 for m_n in matches if len(m_n) == 2 and m_n[0].distance < RATIO_THRESH * m_n[1].distance)

def find_intersection(pos_matches, neg_matches):
    if not pos_matches or not neg_matches:
        return 10.0
    
    max_pos = max(pos_matches)
    max_neg = max(neg_matches)
    min_pos = min(pos_matches)
    
    if max_neg < min_pos:
        return (max_neg + min_pos) / 2.0
        
    max_val = max(max_pos, max_neg)
    bins = np.arange(0, max_val + 2)
    
    hist_pos, _ = np.histogram(pos_matches, bins=bins, density=True)
    hist_neg, _ = np.histogram(neg_matches, bins=bins, density=True)
    
    window = np.ones(3) / 3.0
    hist_pos = np.convolve(hist_pos, window, mode='same')
    hist_neg = np.convolve(hist_neg, window, mode='same')
    
    diff = hist_pos - hist_neg
    idx = np.argwhere(np.diff(np.sign(diff))).flatten()
    
    if len(idx) > 0:
        mid_mean = (np.mean(pos_matches) + np.mean(neg_matches)) / 2.0
        best_idx = min(idx, key=lambda i: abs(i - mid_mean))
        return float(best_idx)
    else:
        return (max_neg + min_pos) / 2.0 if min_pos > max_neg else max_neg + 1.0

def plot_histograms(pos_matches, neg_matches, threshold, title):
    plt.figure(figsize=(8, 4))
    
    max_val = max(max(pos_matches, default=0), max(neg_matches, default=0))
    bins = np.arange(0, max_val + 2)
    
    hist_pos, _ = np.histogram(pos_matches, bins=bins, density=True)
    hist_neg, _ = np.histogram(neg_matches, bins=bins, density=True)
    
    window = np.ones(3) / 3.0
    hist_pos_smooth = np.convolve(hist_pos, window, mode='same')
    hist_neg_smooth = np.convolve(hist_neg, window, mode='same')
    
    plt.hist(neg_matches, bins=bins, density=True, alpha=0.3, color='blue', label='Negative Matches')
    plt.hist(pos_matches, bins=bins, density=True, alpha=0.3, color='red', label='Positive Matches')
    
    plt.plot(bins[:-1], hist_neg_smooth, color='blue', label='Negative Smooth (PDF)')
    plt.plot(bins[:-1], hist_pos_smooth, color='red', label='Positive Smooth (PDF)')
    
    plt.axvline(threshold, color='green', linestyle='dashed', linewidth=2, label=f'Threshold: {threshold:.1f}')
    plt.title(title)
    plt.xlabel('Match Count')
    plt.ylabel('Density (PDF)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def get_db_features(png_dir, db_dir):
    if not os.path.exists(db_dir):
        os.makedirs(db_dir)
        
    db_features = {}
    
    npy_files = glob.glob(os.path.join(db_dir, "sift_database", "*.npy"))
    if npy_files:
        print(f"--- Cargando características precalculadas desde {db_dir} ---")
        for f in npy_files:
            des = np.load(f)
            png_name = os.path.basename(f).replace(".npy", ".png")
            db_features[png_name] = des
        return db_features
        
    print(f"--- Extrayendo y guardando características en {db_dir} ---")
    png_files = glob.glob(os.path.join(png_dir, "*.png"))
    for f in png_files:
        _, des = extract_features(f)
        if des is not None:
            filename = os.path.basename(f)
            db_features[filename] = des
            npy_path = os.path.join(db_dir, filename.replace(".png", ".npy"))
            np.save(npy_path, des)
            
    return db_features

def train_thresholds(train_set, db_features):
    if os.path.exists(THRESHOLDS_FILE):
        print(f"--- Cargando umbrales desde {THRESHOLDS_FILE} ---")
        with open(THRESHOLDS_FILE, "r") as f:
            return json.load(f)
            
    print("--- Entrenando el modelo con imágenes completas (Histogramas) ---")
    
    train_features = {}
    for cat, paths in train_set.items():
        train_features[cat] = []
        for p in paths:
            _, des = extract_features(p)
            if des is not None:
                train_features[cat].append((p, des))

    thresholds = {}
    regions_to_plot = 3
    plots_done = 0
    
    for db_file, des_db in sorted(db_features.items()):
        filename = os.path.basename(db_file)
        comp_category = get_category_from_db_filename(filename)
        denomination = get_denomination_from_category(comp_category)

        positive_matches = []
        negative_matches = []

        for cat, features in train_features.items():
            for p, des_query in features:
                matches = count_good_matches(des_query, des_db)
                if denomination.startswith(cat) and len(cat) <= len(denomination):
                    positive_matches.append(matches)
                else:
                    negative_matches.append(matches)
        
        threshold = find_intersection(positive_matches, negative_matches)
        thresholds[filename] = threshold
        
        if plots_done < regions_to_plot and positive_matches and max(positive_matches) > 30:
            plot_histograms(positive_matches, negative_matches, threshold, f"Histogram Intersection: {filename}")
            plots_done += 1

    with open(THRESHOLDS_FILE, "w") as f:
        json.dump(thresholds, f, indent=4)
        
    print(f"Entrenamiento completado y guardado en {THRESHOLDS_FILE}. Umbrales calculados para {len(thresholds)} billetes completos.")
    return thresholds

def evaluate(test_set, db_features, thresholds):
    print("--- Evaluando el modelo con imágenes de prueba ---")
    correct = 0
    total = 0
    
    for true_cat, paths in test_set.items():
        for p in paths:
            _, des_query = extract_features(p)
            if des_query is None: continue
            
            matched_regions = defaultdict(int)
            total_matches = defaultdict(int)

            for db_file, des_db in db_features.items():
                filename = os.path.basename(db_file)
                category = get_category_from_db_filename(filename)
                denomination = get_denomination_from_category(category)
                
                good_matches = count_good_matches(des_query, des_db)
                T_ji = thresholds.get(filename, 10)
                
                if good_matches >= T_ji:
                    matched_regions[denomination] += 1
                    total_matches[denomination] += good_matches

            valid_candidates = {cat: count for cat, count in matched_regions.items() if count >= 1}
            
            predicted_cat = None
            if valid_candidates:
                results = sorted(valid_candidates.items(), key=lambda x: (x[1], total_matches[x[0]]), reverse=True)
                predicted_cat = results[0][0]
                
            is_correct = False
            if predicted_cat is not None:
                base_pred = predicted_cat.replace("Polimero", "")
                is_correct = (base_pred == true_cat)
                
            if is_correct: correct += 1
            total += 1
            
            print(f"Imagen: {os.path.basename(p)} | Real: {true_cat} | Predicción: {predicted_cat} | {'CORRECTO' if is_correct else 'INCORRECTO'}")
            
    accuracy = correct / total if total > 0 else 0
    print("-" * 40)
    print(f"Precisión Global (Accuracy): {accuracy * 100:.2f}% ({correct}/{total})")
    print("-" * 40)

# Ejecución principal
dataset = load_dataset(test_images_dir)
train_set, test_set = split_dataset(dataset, train_ratio=0.75)

print("Imágenes de entrenamiento por categoría:", {k: len(v) for k, v in train_set.items()})
print("Imágenes de prueba por categoría:", {k: len(v) for k, v in test_set.items()})

db_features = get_db_features(png_dir, database_dir)
thresholds = train_thresholds(train_set, db_features)
evaluate(test_set, db_features, thresholds)


Imágenes de entrenamiento por categoría: {'100': 12, '20': 17, '50': 8, '1000': 15, '200': 7, '500': 12}
Imágenes de prueba por categoría: {'100': 5, '20': 6, '50': 3, '1000': 5, '200': 3, '500': 5}
--- Cargando características precalculadas desde ../data/full_bill_database/ ---
--- Cargando umbrales desde thresholds.json ---
--- Evaluando el modelo con imágenes de prueba ---
Imagen: 9.jpeg | Real: 100 | Predicción: 100 | CORRECTO
Imagen: 13.jpeg | Real: 100 | Predicción: 100 | CORRECTO
Imagen: 4.jpeg | Real: 100 | Predicción: 100 | CORRECTO
Imagen: 1.jpeg | Real: 100 | Predicción: 100 | CORRECTO
Imagen: 12.jpeg | Real: 100 | Predicción: 500 | INCORRECTO
Imagen: 5.jpeg | Real: 20 | Predicción: 20Polimero | CORRECTO
Imagen: 16.jpeg | Real: 20 | Predicción: 20Polimero | CORRECTO
Imagen: 17.jpeg | Real: 20 | Predicción: 20 | CORRECTO
Imagen: 1.jpeg | Real: 20 | Predicción: 20Polimero | CORRECTO
Imagen: 12.jpeg | Real: 20 | Predicción: 20 | CORRECTO
Imagen: 7.jpeg | Real: 20 | Predicción: 